# AquaLens 2030
### Saudi Urban Water Source Intelligence Platform

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/98raneemsaif-create/aqualens-2030/blob/main/notebooks/aqualens_2030_colab_demo.ipynb)

AquaLens 2030 explores **water-source concentration across Saudi urban regions** using official GASTAT data, a Delta Lakehouse, data-quality controls, orchestration, lineage, and an official-document RAG pipeline.

The project separates two questions:

- **Lakehouse analytics:** *What does the data show?*
- **Grounded RAG:** *What strategic or methodological context do official Saudi sources provide?*

Water-source concentration is treated as a descriptive data characteristic, **not as a risk classification**.

This notebook is the portable walkthrough of the project. Run it top to bottom in Google Colab to reproduce the components that fit naturally in a notebook: source-data checks, Pydantic validation, Delta transformations, Great Expectations, hybrid retrieval, and optional Gemini generation. Kafka, Airflow, and OpenLineage require the full Docker runtime, so their sections load the actual run artifacts committed with the repository instead of replacing those systems with notebook mocks.

**Runtime note:** first-time installation and CPU model downloads may take several minutes and roughly 1 GB of model storage. Gemini is optional and disabled by default.

## 1 — Architecture and data flow

AquaLens combines an analytical Lakehouse path with a separate official-document RAG path.

```text
ANALYTICAL DATA PATH
────────────────────
Official GASTAT CSV
        │
        ▼
Kafka producer ──► raw topic
        │
        ▼
Pydantic validation
   │            │
valid           malformed
   │            └──► Kafka quarantine / DLQ
   ▼
Delta Bronze (append-only)
   ▼
Delta Silver (normalized + MERGE on year + region + source)
   ▼
Great Expectations quality gate
   ▼
Delta Gold
   ▼
Regional water-source concentration profiles


OFFICIAL-DOCUMENT CONTEXT PATH
──────────────────────────────
MEWA National Water Strategy 2030
GASTAT Water Accounts Methodology
        │
        ▼
Extraction → deterministic chunking
        ▼
E5 embeddings + Chroma
        │
        ├──► dense retrieval
        └──► BM25 retrieval
                 │
                 ▼
              RRF fusion
                 ▼
        CrossEncoder reranking
                 ▼
      Gemini grounded answer
       + source citations
```

Airflow orchestrates the end-to-end project stages, while OpenLineage records `START`, `COMPLETE`, and `FAIL` lifecycle events.

The two paths are complementary: the Lakehouse describes **what the data shows**, while RAG provides **official strategic and methodological context**. Gold analytical values are not silently injected into the document-grounded answer.

## 2 — Repository setup

Clone the repository and move into its root so every path in the notebook remains relative and reproducible. The printed Git revision makes it easy to identify the exact project version used for the run.

In [ ]:
import os, sys, json, csv, hashlib, subprocess, uuid, html
from pathlib import Path
from IPython.display import display, Markdown, JSON, HTML
REPOSITORY = "https://github.com/98raneemsaif-create/aqualens-2030"
repo = Path("aqualens-2030")
if not Path("AGENTS.md").exists():
    if not repo.exists():
        subprocess.run(["git", "clone", "--quiet", REPOSITORY, str(repo)], check=True)
    os.chdir(repo)
assert Path("AGENTS.md").exists(), "Run from the cloned AquaLens repository."
print("Repository revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
WORK = Path("storage/colab") / str(uuid.uuid4())
WORK.mkdir(parents=True, exist_ok=False)
def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))
def table(rows):
    if not rows:
        print("No rows"); return
    keys = list(rows[0])
    display(HTML("<table><tr>" + "".join("<th>"+html.escape(str(k))+"</th>" for k in keys) + "</tr>" +
                 "".join("<tr>"+"".join("<td>"+html.escape(str(r.get(k,"")))+"</td>" for k in keys)+"</tr>" for r in rows) + "</table>"))
RAG_ARTIFACTS = Path("docs/evidence/phase_d/402947a2-1f21-445a-82a0-e0e3e4be3b70")
PIPELINE_ARTIFACTS = Path("docs/evidence/phase_e")
for required in [RAG_ARTIFACTS / "chunks.json", PIPELINE_ARTIFACTS / "final_success/states.json", PIPELINE_ARTIFACTS / "final_quality_failure/states.json"]:
    assert required.exists(), f"Required committed evidence missing: {required}. Use the completed repository revision."

## 3 — Portable Colab environment

The project runtime is based on **Python 3.11**. Current Google Colab images may use a newer system Python and may not include the OS package required by the standard `venv` module, so this notebook does not depend on Colab's built-in `python -m venv`.

Instead, the cell below uses the lightweight `uv` bootstrap tool to provision an isolated **Python 3.11** environment under `storage/colab/venv`, then installs only the repository packages required by the live notebook workflow with versions aligned to `requirements/runtime.lock`.

This keeps the notebook compatible with Colab while leaving both the repository dependency files and Colab's base Python environment unchanged.

The notebook does **not** install Airflow or launch Kafka/OpenLineage services. Those belong to the project's Docker runtime and are covered later through the stored project run artifacts.

PDF extraction libraries are not reinstalled here because the project already preserves the finalized deterministic 445-chunk corpus and source metadata. This keeps the Colab path faster while retaining the same RAG inputs used by the completed project.

In [ ]:
import shutil

# Read direct package versions from the project's frozen runtime lock.
pins = {}
for raw_line in Path("requirements/runtime.lock").read_text(encoding="utf-8").splitlines():
    line = raw_line.strip()
    if line and not line.startswith("#") and "==" in line:
        name, version = line.split("==", 1)
        pins[name.lower()] = version

packages = [
    "pydantic", "deltalake", "pyarrow", "pandas", "numpy",
    "great-expectations", "chromadb", "sentence-transformers",
    "rank-bm25", "google-genai", "transformers", "sentencepiece",
]
required = ["torch", *packages]
missing_pins = [p for p in required if p.lower() not in pins]
assert not missing_pins, f"Missing required pins in runtime.lock: {missing_pins}"

ENV = Path("storage/colab/venv")
PY = ENV / "bin/python"

def environment_ready():
    if not PY.exists():
        return False
    probe = subprocess.run(
        [str(PY), "-c", "import sys, pip; assert sys.version_info[:2] == (3, 11)"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    return probe.returncode == 0

if not environment_ready():
    # Remove any incomplete environment left by a previous interrupted/failed cell.
    shutil.rmtree(ENV, ignore_errors=True)

    # Colab may not ship the OS package needed by `python -m venv`.
    # uv can provision the project's Python 3.11 runtime without apt/system changes.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "uv"],
        check=True,
    )
    UV = shutil.which("uv")
    assert UV, "uv was installed but its executable was not found."

    # Ensure Python 3.11 is available, then create a seeded venv with pip.
    subprocess.run([UV, "python", "install", "3.11"], check=True)
    subprocess.run(
        [UV, "venv", "--python", "3.11", "--seed", str(ENV)],
        check=True,
    )

assert environment_ready(), "The isolated Python 3.11 Colab environment could not be created."

# Install the live notebook dependency set. PyPI remains the primary index;
# the PyTorch CPU index is an extra source so Colab does not need GPU wheels.
install_args = (
    [str(PY), "-m", "pip", "install", "--quiet",
     "--index-url", "https://pypi.org/simple",
     "--extra-index-url", "https://download.pytorch.org/whl/cpu",
     "--constraint", "requirements/runtime.lock"]
    + ["torch==" + pins["torch"]]
    + [p + "==" + pins[p.lower()] for p in packages]
)
subprocess.run(install_args, check=True)
subprocess.run([str(PY), "-m", "pip", "check"], check=True)

runtime_version = subprocess.check_output(
    [str(PY), "-c", "import platform; print(platform.python_version())"],
    text=True,
).strip()
print(f"Portable AquaLens runtime ready: Python {runtime_version}")
print("Repository dependency files and Colab base environment remain unchanged.")

def run_project(source, secret=None):
    env = os.environ.copy()
    existing_pythonpath = env.get("PYTHONPATH", "")
    repo_path = str(Path.cwd())
    env.update({
        "PYTHONPATH": repo_path + (os.pathsep + existing_pythonpath if existing_pythonpath else ""),
        "COLAB_WORK": str(WORK.resolve()),
        "HF_HOME": str(Path("storage/colab/models").resolve()),
        "OMP_NUM_THREADS": "1",
        "MKL_NUM_THREADS": "1",
        "TOKENIZERS_PARALLELISM": "false",
    })
    for key in [
        "GEMINI_API_KEY", "GOOGLE_API_KEY",
        "GOOGLE_GENAI_USE_VERTEXAI", "GOOGLE_GENAI_USE_ENTERPRISE",
        "GOOGLE_GEMINI_BASE_URL", "GOOGLE_VERTEX_BASE_URL",
    ]:
        env.pop(key, None)

    prelude = "import os,json; from pathlib import Path; W=Path(os.environ['COLAB_WORK'])\n"
    result = subprocess.run(
        [str(PY), "-c", prelude + source],
        input=secret or "",
        text=True,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    if result.returncode:
        # Show a short, secret-safe diagnostic instead of hiding the useful exception.
        diagnostic = result.stderr.strip().splitlines()
        diagnostic = "\n".join(diagnostic[-12:])
        raise RuntimeError(
            f"Live project cell failed (exit {result.returncode}).\n{diagnostic}"
        )

    if secret is None and result.stdout.strip():
        print(result.stdout[-3000:])

## 4 — Official urban water dataset

The analytical source is **Water Distribution in Urban Sector by Source — Annual**, published by GASTAT and obtained through DataSaudi.

The source snapshot contains:

- 56 observations for 2024
- 13 Saudi regions plus `Grand Total`
- four water-source categories
- four `Grand Total` rows
- 20 valid zero-volume observations

The project business key is:

`year + region + source`

The SHA-256 check below protects the local project snapshot from accidental modification.

In [ ]:
SOURCE = Path("data/source/water_distribution_urban_saudi.csv")
source_hash = hashlib.sha256(SOURCE.read_bytes()).hexdigest()
assert source_hash == "4df640b65d3341c1e42e64be7582434aa5e19ceabfa952feb35195f8350a849c"

with SOURCE.open(encoding="utf-8", newline="") as f:
    source_rows = list(csv.DictReader(f))

business_keys = [(int(r["Year"]), r["Province"], r["Source"]) for r in source_rows]
source_categories = sorted({r["Source"] for r in source_rows})

metrics = {
    "source rows": len(source_rows),
    "years": sorted({int(r["Year"]) for r in source_rows}),
    "real regions": len({r["Province"] for r in source_rows if r["Province"] != "Grand Total"}),
    "source categories": source_categories,
    "Grand Total rows": sum(r["Province"] == "Grand Total" for r in source_rows),
    "zero rows": sum(float(r["Value"]) == 0 for r in source_rows),
    "business keys unique": len(business_keys) == len(set(business_keys)),
}

assert metrics["source rows"] == 56
assert metrics["years"] == [2024]
assert metrics["real regions"] == 13
assert len(metrics["source categories"]) == 4
assert metrics["Grand Total rows"] == 4
assert metrics["zero rows"] == 20
assert metrics["business keys unique"]

display(JSON(metrics))
print("Source SHA-256:", source_hash)
table(source_rows[:4])

## 5 — Ingestion schema validation with Pydantic

The project validates message structure and types at the ingestion boundary with `WaterEvent`.

This distinction is intentional:

- `0` is a valid water-volume value.
- a numeric negative value is structurally valid and is allowed to reach the later business-quality gate.
- a malformed numeric value such as `"not-a-number"` is rejected by Pydantic.

This cell exercises the actual project model without simulating Kafka.

In [ ]:
run_project(r"""
from src.ingestion.schema import WaterEvent
from pydantic import ValidationError
base=dict(year=2024,region='Al-Riyadh',source='Groundwater',volume_m3=1.0,
          run_id='colab-structural-test',event_id='test-only',record_kind='test_fixture')
results=[]
for label,value in [('normal',1.0),('zero',0.0),('numeric negative',-1.0),('malformed','not-a-number')]:
    try:
        WaterEvent.model_validate(dict(base,volume_m3=value)); accepted=True
    except ValidationError:
        accepted=False
    assert accepted == (label!='malformed')
    results.append(dict(case=label,accepted=accepted))
(W/'pydantic.json').write_text(json.dumps(results))
""")
table(read_json(WORK / "pydantic.json"))

## 6 — Kafka ingestion and quarantine

Kafka is part of the full Docker runtime, so it is not started inside Colab. Instead, this section reads the recorded result of the project's real Kafka producer/consumer run.

The run published the official 56 source observations plus one deliberately malformed test event. Valid events were accepted; the malformed event was routed to the real quarantine topic with its rejection reason. The second run also demonstrates that persistent Kafka history does not contaminate run-specific counts.

In [ ]:
kafka = read_json("docs/evidence/phase_b/proof/summary.json")
isolation = read_json("docs/evidence/phase_b/isolation_audit.json")
display(JSON({"produced": kafka["producer"]["produced_count"], "accepted": kafka["consumer"]["accepted_count"],
              "quarantined": kafka["consumer"]["quarantined_count"], "raw topic": kafka["producer"]["raw_topic"],
              "quarantine topic": kafka["producer"]["quarantine_topic"]}))
print(kafka["verification"]["quarantine_records_read_from_kafka"][0]["rejection_reason"])
display(JSON(isolation))
assert isolation["raw_offsets_disjoint"]

## 7 — Delta Lakehouse: Bronze, Silver, and Gold

A fresh clone does not contain the gitignored Kafka staging files, so the Colab walkthrough maps the immutable official CSV into the same project event schema and creates temporary staging under `storage/colab/`.

The transformations themselves use the **actual project Lakehouse functions**:

- **Bronze:** append-only Delta storage
- **Silver:** normalized records with a real Delta `MERGE` on `year + region + source`
- **Schema enforcement:** an incompatible Delta write is intentionally rejected
- **Gold:** one regional water-source profile per region

`Grand Total` remains in Bronze and Silver and is excluded only from regional Gold. Zero-volume observations are preserved.

In [ ]:
run_project(r"""
import csv,uuid
from src.ingestion.schema import WaterEvent
from src.lakehouse.bronze import append_bronze,read_accepted
from src.lakehouse.silver import merge_silver
from src.lakehouse.gold import build_gold
from src.lakehouse.schema_proof import prove_schema_rejection
from deltalake import DeltaTable
root=W/'delta'/str(uuid.uuid4()); root.mkdir(parents=True)
with open('data/source/water_distribution_urban_saudi.csv') as f:
    rows=[WaterEvent(year=int(r['Year']),region=r['Province'],source=r['Source'],volume_m3=float(r['Value']),
                    run_id='colab-source-replay',event_id=f'source-{i:04d}',record_kind='source').model_dump()
          for i,r in enumerate(csv.DictReader(f),1)]
staging=root/'accepted.jsonl'; staging.write_text(''.join(json.dumps(r)+'\n' for r in rows))
bronze=root/'bronze';silver=root/'silver';gold=root/'gold_regional_water_profile'
append_bronze(staging,bronze)
first=merge_silver(bronze,silver);second=merge_silver(bronze,silver)
b=DeltaTable(bronze).to_pyarrow_table().to_pylist();s=DeltaTable(silver).to_pyarrow_table().to_pylist()
assert len(b)==len(s)==56 and len({(r['year'],r['region'],r['source']) for r in s})==56
assert all(sum(r['region']=='Grand Total' for r in data)==4 and sum(r['volume_m3']==0 for r in data)==20 for data in [b,s])
schema=prove_schema_rejection(root/'schema_proof',read_accepted(staging));assert schema['unchanged']
build_gold(silver,gold);profiles=DeltaTable(gold).to_pyarrow_table().to_pylist()
assert len(profiles)==13 and all(r['region']!='Grand Total' for r in profiles)
(W/'delta.json').write_text(json.dumps(dict(root=str(root),silver=str(silver),bronze_rows=len(b),silver_rows=len(s),gold_rows=len(profiles),
 first_merge=first,second_merge=second,silver_history=DeltaTable(silver).history(),schema_proof=schema,gold=profiles)))
""")
delta = read_json(WORK / "delta.json")
display(JSON({k:delta[k] for k in ["bronze_rows","silver_rows","gold_rows","first_merge","second_merge"]}))
display(JSON(delta["schema_proof"]))
table(delta["gold"])

## 8 — Source provenance reconciliation

The source includes supplied `Grand Total` observations. This section compares them with the sum of regional observations without changing the data.

Two one-unit differences are preserved exactly as supplied:

- Desalinated water: regional sum is 1 m³ below the supplied total
- Surface water: regional sum is 1 m³ below the supplied total

Groundwater and Other sources reconcile exactly. No cause is inferred from these differences.

In [ ]:
from decimal import Decimal
reconciliation = []
for source in sorted({r["Source"] for r in source_rows}):
    rows = [r for r in source_rows if r["Source"] == source]
    regional = sum(Decimal(r["Value"]) for r in rows if r["Province"] != "Grand Total")
    supplied = sum(Decimal(r["Value"]) for r in rows if r["Province"] == "Grand Total")
    reconciliation.append({"source":source,"regional m3":str(regional),"supplied m3":str(supplied),"difference m3":str(regional-supplied)})
table(reconciliation)

## 9 — Data quality with Great Expectations

The project applies Great Expectations to Silver before Gold is allowed to run.

This notebook demonstrates both outcomes with the actual `validate_silver` implementation:

1. the official Silver dataset passes;
2. a separate controlled test fixture with `volume_m3 = -1` fails the nonnegative-volume expectation.

The negative fixture is test-only data and is never written into the official analytical source.

In [ ]:
run_project(r"""
import uuid
import pyarrow as pa
from deltalake import DeltaTable,write_deltalake
from src.lakehouse.bronze import EVENT_SCHEMA
from src.quality.gate import validate_silver,DataQualityError
delta=json.loads((W/'delta.json').read_text());root=W/'gx'/str(uuid.uuid4());root.mkdir(parents=True)
passed=validate_silver(Path(delta['silver']),root/'success.json');assert passed['success']
rows=DeltaTable(delta['silver']).to_pyarrow_table().to_pylist()
fixture=json.loads(Path('tests/fixtures/negative_water_event.json').read_text())
fixture.update(run_id='colab-quality-test',event_id='negative-test-only',record_kind='test_fixture')
write_deltalake(root/'negative',pa.Table.from_pylist(rows+[fixture],schema=EVENT_SCHEMA))
try:
    validate_silver(root/'negative',root/'failure.json')
except DataQualityError:
    failure=json.loads((root/'failure.json').read_text())
else:
    raise AssertionError('GX unexpectedly accepted negative volume')
failed=[r for r in failure['results'] if not r['success']]
assert any(r['expectation_config']['kwargs'].get('column')=='volume_m3' and -1.0 in r['result'].get('partial_unexpected_list',[]) for r in failed)
(W/'gx.json').write_text(json.dumps(dict(official_passed=True,controlled_failure_observed=True,fixture=fixture,failed_expectations=failed)))
""")
gx = read_json(WORK / "gx.json"); display(JSON(gx))

## 10 — Official-document hybrid RAG

The RAG corpus is built from two official Saudi sources:

- **National Water Strategy 2030 — MEWA**
- **Methodology and Quality Report of Water Accounts — GASTAT**

The repository preserves **445 deterministic chunks** with source metadata, physical page identity, and source hashes. Reusing this finalized corpus keeps the Colab run faster and ensures retrieval uses the same document representation as the completed project.

During project development, extraction quality was checked explicitly. MEWA's Arabic narrative text uses PyMuPDF's default unsorted text extraction because sorted/layout modes disrupted Arabic reading order; unreliable numeric/table-like fragments are excluded from the RAG corpus. GASTAT's English methodology document uses the verified pypdf extraction path.

The retrieval stack is:

- `intfloat/multilingual-e5-small` — normalized 384-dimensional dense embeddings
- ChromaDB `PersistentClient`
- BM25 keyword retrieval
- dense top 8 + BM25 top 8
- Reciprocal Rank Fusion with `k=60`
- `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1`
- final top evidence chunks for grounded generation

`RUN_LIVE_RETRIEVAL = True` rebuilds a temporary local Chroma collection and executes the real project retriever. Set it to `False` when you only want to inspect the stored project retrieval output.

In [ ]:
corpus = read_json(RAG_ARTIFACTS / "chunks.json")
assert len(corpus["chunks"]) == 445
for name in ["mewa_strategy", "gastat_methodology"]:
    meta = read_json(Path("rag/sources") / (name + ".json"))
    assert hashlib.sha256((Path("rag/sources") / meta["filename"]).read_bytes()).hexdigest() == meta["sha256"]
table(corpus["documents"])
display(JSON(corpus["chunks"][0]["metadata"]))
RUN_LIVE_RETRIEVAL = True  # Change to False to display the stored project retrieval output only.
if RUN_LIVE_RETRIEVAL:
    run_project(r"""
import uuid,numpy as np
from src.rag.retrieval import Retriever
p=Path('docs/evidence/phase_d/402947a2-1f21-445a-82a0-e0e3e4be3b70')
chunks=json.loads((p/'chunks.json').read_text())['chunks'];query=json.loads((p/'query_1_retrieval.json').read_text())['query']
r=Retriever(chunks,W/'chroma'/str(uuid.uuid4()))
assert r.collection.count()==445 and r.vectors.shape==(445,384)
assert np.allclose(np.linalg.norm(r.vectors,axis=1),1,atol=1e-5)
(W/'retrieval.json').write_text(json.dumps(r.retrieve(query),ensure_ascii=False),encoding='utf-8')
""")
    retrieval = read_json(WORK / "retrieval.json")
    retrieval_origin = "Live Colab project Retriever"
else:
    retrieval = read_json(RAG_ARTIFACTS / "query_1_retrieval.json")
    retrieval_origin = "Stored Phase D retrieval output"
print(retrieval_origin); print(retrieval["query"])
for label in ["dense", "bm25", "fused", "reranked"]:
    display(Markdown("**" + label + "**")); table(retrieval[label])
for chunk in retrieval["final_chunks"]:
    display(JSON(chunk["metadata"])); print(chunk["text"])

## 11 — Optional grounded generation with Gemini

Live generation is optional so the notebook can run without storing or requiring a personal API key.

If `ENABLE_LIVE_GEMINI = True`, the notebook requests the key interactively with `getpass` and passes it only to the isolated subprocess. The key is not written to the notebook, repository, command line, or output.

Generation uses the project's current configuration:

- `gemini-3.5-flash`
- temperature `0`
- retrieved official context only
- traceable document/page/URL citations
- bounded retry handling for transient API failures

When live generation is skipped or temporarily unavailable, the notebook displays the grounded answer and citations already produced by the completed project run.

In [ ]:
ENABLE_LIVE_GEMINI = False  # Optional; no API key is required for the rest of the notebook.
fallback = read_json("docs/evidence/rag/airflow_be770adc-2479-456e-aa2f-2715eac4f484.json")
answer = fallback["answer"]
answer_context = fallback["retrieval"]["final_chunks"]
answer_origin = "Stored project answer (no new API call)"
if ENABLE_LIVE_GEMINI:
    from getpass import getpass
    api_key = getpass("Optional GEMINI_API_KEY (blank skips live generation): ").strip()
    if api_key:
        try:
            (WORK / "selected_context.json").write_text(json.dumps(retrieval, ensure_ascii=False), encoding="utf-8")
            run_project(r"""
import sys
from src.rag.generation import generate,MODEL
assert MODEL=='gemini-3.5-flash'
os.environ['GEMINI_API_KEY']=sys.stdin.read()
try:
    r=json.loads((W/'selected_context.json').read_text(encoding='utf-8'))
    result=generate(r['query'],r['final_chunks'])
    (W/'live_answer.json').write_text(json.dumps(result,ensure_ascii=False),encoding='utf-8')
except Exception:
    raise SystemExit(1)
finally:
    os.environ.pop('GEMINI_API_KEY',None)
""", secret=api_key)
            answer = read_json(WORK / "live_answer.json")
            answer_context = retrieval["final_chunks"]
            answer_origin = "Live Gemini generation"
        except Exception:
            print("Live Gemini unavailable; displaying the stored project answer instead.")
        finally:
            del api_key
    else:
        del api_key
allowed = {c["chunk_id"]:c for c in answer_context}
for citation in answer["citations"]:
    assert citation["chunk_id"] in allowed
    assert all(citation[k] == allowed[citation["chunk_id"]]["metadata"][k] for k in ["title", "page", "canonical_url"])
print(answer_origin); print(answer["answer"])
table([{k:c[k] for k in ["title", "page", "canonical_url", "chunk_id"]} for c in answer["citations"]])

## 12 — End-to-end orchestration with Airflow

The full project uses Airflow to connect the pipeline stages:

`produce_events → consume_and_validate → bronze_load → silver_merge → schema_enforcement_proof → quality_gate → gold_build → rag_chunk_and_index → rag_grounded_answer_smoke_test`

Running Airflow inside Colab would add infrastructure without improving the notebook walkthrough, so this section loads the recorded states from the actual Docker execution.

Two runs are shown:

- `phase_e_success`: all nine tasks completed successfully.
- `phase_e_quality_failure`: the controlled negative-volume fixture caused the Great Expectations gate to fail, and Gold/RAG were marked `upstream_failed` without executing.

The failed DAG state in the second run is intentional and demonstrates quality-gate blocking behavior.

In [ ]:
success = read_json(PIPELINE_ARTIFACTS / "final_success/states.json")
failure = read_json(PIPELINE_ARTIFACTS / "final_quality_failure/states.json")
assert success["run_id"] == "phase_e_success" and success["state"] == "success"
assert len(success["tasks"]) == 9 and all(t["state"] == "success" for t in success["tasks"])
assert failure["run_id"] == "phase_e_quality_failure" and failure["state"] == "failed"
assert failure["conf"] == {"quality_failure_demo": True}
expected = {t["task_id"]:"success" for t in success["tasks"]}
expected.update(quality_gate="failed",gold_build="upstream_failed",rag_chunk_and_index="upstream_failed",rag_grounded_answer_smoke_test="upstream_failed")
assert {t["task_id"]:t["state"] for t in failure["tasks"]} == expected
for run in [success, failure]:
    print(run["run_id"], "->", run["state"])
    table([{k:t[k] for k in ["task_id","state","try_number"]} for t in run["tasks"]])

## 13 — Pipeline lineage with OpenLineage

The Docker runtime uses the Airflow OpenLineage provider with FileTransport.

This section summarizes the emitted lifecycle events from the recorded project runs:

- successful stages emit `START` and `COMPLETE`;
- failed stages emit `START` and `FAIL`;
- downstream task bodies blocked by the quality gate have no execution events.

Historical Gemini service failures are retained in the lineage history alongside the later successful completion rather than being removed.

In [ ]:
lineage_dashboard = []
for label in ["success", "failure"]:
    summary = read_json(PIPELINE_ARTIFACTS / ("final_lineage_" + label) / "summary.json")
    stages = [r for r in summary["records"] if r["job"].startswith("aqualens_2030_pipeline.")]
    lineage_dashboard.append({"run":summary["run_id"], **{kind:sum(r["event_type"]==kind for r in stages) for kind in ["START","COMPLETE","FAIL"]}})
    if label == "failure":
        blocked = {"gold_build", "rag_chunk_and_index", "rag_grounded_answer_smoke_test"}
        assert not any(r["job"].split(".")[-1] in blocked for r in stages)
        selected = [r for r in stages if r["job"].endswith(".quality_gate") or r["event_type"]=="COMPLETE"]
    else:
        selected = [r for r in stages if r["job"].endswith(".rag_grounded_answer_smoke_test")]
    table([{k:r[k] for k in ["job","event_type","attempt","file"]} for r in selected])
table(lineage_dashboard)

## 14 — Project results

This final view brings together the live portable outputs and the full-runtime project results:

- source-data integrity
- Bronze / Silver / Gold row counts
- Great Expectations success and controlled failure
- RAG corpus and retrieval output
- grounded answer and citations
- Airflow success and quality-blocking paths
- OpenLineage lifecycle summary

It provides one concise view of how the components work together while the preceding sections retain the implementation detail.

In [ ]:
assert hashlib.sha256(SOURCE.read_bytes()).hexdigest() == source_hash
table([{"measure":k,"result":v} for k,v in {
    "Source records":len(source_rows),"Bronze records (live)":delta["bronze_rows"],
    "Silver records (live)":delta["silver_rows"],"Regional Gold profiles (live)":delta["gold_rows"],
    "GX official pass (live)":gx["official_passed"],"GX negative fixture rejected (live)":gx["controlled_failure_observed"],
    "RAG corpus chunks":len(corpus["chunks"]),"Retrieval origin":retrieval_origin,
    "Answer origin":answer_origin,"Citations resolved":len(answer["citations"]),
    "Airflow success run":success["state"],"Controlled quality-failure run":failure["state"]}.items()])
table(delta["gold"][:3]); table(lineage_dashboard)
print("Example query:", retrieval["query"])
print(answer["answer"])

## 15 — Reproducibility and project artifacts

The notebook uses the same code, data contracts, models, and transformation rules as the main repository. Temporary Colab outputs are written under gitignored `storage/colab/` and disappear with the Colab session.

For deeper inspection, implementation code and run artifacts are organized by component:

| Component | Main repository locations |
|---|---|
| Official analytical data | `data/source/water_distribution_urban_saudi.csv` |
| Kafka ingestion, validation, quarantine | `src/ingestion/`, `docs/evidence/phase_b/` |
| Delta Bronze / Silver / Gold | `src/lakehouse/`, `docs/evidence/phase_c/` |
| Great Expectations quality gate | `src/quality/`, `docs/evidence/quality/`, `docs/evidence/phase_e/` |
| Official RAG sources | `rag/sources/` |
| Chunking, retrieval, generation | `src/rag/`, `docs/evidence/phase_d/`, `docs/evidence/rag/` |
| Airflow orchestration | `dags/aqualens_pipeline.py`, `docs/evidence/phase_e/` |
| OpenLineage events | `docs/evidence/lineage/raw/`, `docs/evidence/phase_e/final_lineage_success/`, `docs/evidence/phase_e/final_lineage_failure/` |
| Environment and dependencies | `Dockerfile`, `docker-compose.yml`, `requirements/` |
| Project documentation | `README.md`, `docs/` |

### Running options

**Google Colab**
- Best for the portable technical walkthrough in this notebook.
- No Docker is required.
- Internet is required for the one-time Python 3.11/package bootstrap and model downloads.
- Gemini remains optional.

**Full local project**
- Use the Docker Compose environment described in the repository README.
- This is the path for running Kafka, Airflow, and OpenLineage together.

The official source snapshot and raw RAG documents are never modified by notebook execution.

---
### Author

Raneem Saif Aldawsari  
Data Engineer  

GitHub: https://github.com/98raneemsaif-create  
LinkedIn: https://www.linkedin.com/in/raneem-aldawsari-6a0994262/